# Delta Lake Basics — 03: ACID Transactions

## What you will learn
ACID is what makes Delta different from plain Parquet.
Without it, concurrent writes can corrupt your data silently.

In this notebook:
1. What ACID means in the context of Delta Lake
2. Atomicity — all-or-nothing writes
3. Isolation — concurrent readers always see consistent state
4. Optimistic concurrency — how Delta handles concurrent writers
5. Conflict detection — what fails and what succeeds
6. Retrying on conflicts — production-safe write patterns


In [ ]:
import os, time, pathlib, shutil, random, datetime, subprocess, glob
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
from delta.tables import DeltaTable

MASTER   = 'spark://spark-master:7077'
DATA_DIR = '/workspace/data/delta_basics'
pathlib.Path(DATA_DIR).mkdir(parents=True, exist_ok=True)

spark = (
    SparkSession.builder
    .appName('delta-basics')
    .master(MASTER)
    .config('spark.executor.memory', '2g')
    .config('spark.driver.memory',   '1g')
    .config('spark.executor.cores',  '2')
    .config('spark.shuffle.sort.bypassMergeThreshold', '0')
    .getOrCreate()
)
spark.sparkContext.setLogLevel('WARN')
print(f"Spark {spark.version} | Delta Lake ready")

random.seed(42)
N = 100_000
CATEGORIES = ["Electronics","Clothing","Books","Food","Sports","Furniture"]
COUNTRIES  = ["US","UK","DE","FR","JP","CA","AU","BR"]
STATUSES   = ["pending","confirmed","shipped","delivered","cancelled"]

schema = StructType([
    StructField("order_id",    LongType()),
    StructField("customer_id", LongType()),
    StructField("product",     StringType()),
    StructField("category",    StringType()),
    StructField("country",     StringType()),
    StructField("quantity",    IntegerType()),
    StructField("price",       DoubleType()),
    StructField("revenue",     DoubleType()),
    StructField("status",      StringType()),
    StructField("order_date",  DateType()),
])

base = datetime.date(2024, 1, 1)
rows = []
for i in range(N):
    qty   = random.randint(1, 10)
    price = round(random.uniform(5, 1000), 2)
    d     = base + datetime.timedelta(days=random.randint(0, 364))
    rows.append((i+1, random.randint(1,10000),
                 f"Product_{random.randint(1,200)}",
                 random.choice(CATEGORIES), random.choice(COUNTRIES),
                 qty, price, round(qty*price, 2),
                 random.choice(STATUSES), d))

df = spark.createDataFrame(rows, schema)
df.cache().count()
print(f"Dataset ready: {N:,} rows")

## Step 1 — What ACID Means for Delta Lake

```
A — Atomicity:   A write either fully succeeds or has zero effect.
                 No partial writes visible to readers.

C — Consistency: Schema is enforced. Bad data rejected.
                 Table always in a valid state.

I — Isolation:   Concurrent readers see a consistent snapshot.
                 A reader mid-query is not affected by concurrent writes.

D — Durability:  Once committed, data survives failures.
                 Commit in _delta_log = permanent record.
```

**Why this matters:**
Without ACID (plain Parquet), a writer crash mid-write leaves partial files visible.
Concurrent writers overwrite each other's data.
Schema changes break existing readers immediately.


In [ ]:
TABLE = f"{DATA_DIR}/03_acid"
shutil.rmtree(TABLE, ignore_errors=True)
df.write.format("delta").mode("overwrite").save(TABLE)
print(f"Table created: {spark.read.format('delta').load(TABLE).count():,} rows")

## Step 2 — Atomicity: All-or-Nothing Writes

In Delta, a write is only visible when the commit JSON is written.
If the write crashes partway through, no data is visible — not even partial files.


In [ ]:
import threading, time

# Demonstrate: readers see a consistent snapshot during writes
read_results = []

def continuous_reader():
    """Reads the table every 0.5 seconds while a write is happening."""
    for _ in range(6):
        count = spark.read.format("delta").load(TABLE).count()
        read_results.append(count)
        time.sleep(0.3)

def big_write():
    """Write a large batch — takes time but is atomic."""
    time.sleep(0.1)  # let reader start
    df.write.format("delta").mode("append").save(TABLE)

# Start both concurrently
reader_thread = threading.Thread(target=continuous_reader)
writer_thread = threading.Thread(target=big_write)

reader_thread.start()
writer_thread.start()

reader_thread.join()
writer_thread.join()

print("Row counts seen by reader during concurrent write:")
for i, count in enumerate(read_results):
    print(f"  Read {i+1}: {count:,} rows")

print()
print("Observation: reader sees either the OLD count OR the NEW count.")
print("NEVER a partial/intermediate state — that's atomicity!")

final_count = spark.read.format("delta").load(TABLE).count()
print(f"\nFinal count: {final_count:,} rows")

## Step 3 — Optimistic Concurrency Control

Delta uses **optimistic concurrency** — writes proceed without locking,
but at commit time Delta checks for conflicts with other concurrent writes.

Two writes conflict if they:
1. Modify overlapping data files
2. Write to overlapping partitions in a `MERGE` operation

Non-conflicting writes succeed even if concurrent.


In [ ]:
# Show transaction log to understand conflict detection
print("Transaction log after writes:")
DeltaTable.forPath(spark, TABLE).history() \
    .select("version","timestamp","operation","operationMetrics") \
    .orderBy("version") \
    .show(5, truncate=False)

print("\nConflict detection rules:")
print("  ✅ Append + Append → NO conflict (different files)")
print("  ✅ Append + Read   → NO conflict (readers don't write)")
print("  ⚠️  Overwrite + Append → CONFLICT (overwrite removes files append added)")
print("  ⚠️  replaceWhere overlapping partitions → CONFLICT")
print("  ✅ replaceWhere non-overlapping partitions → NO conflict")

## Step 4 — Schema Consistency: Rejecting Bad Writes


In [ ]:
print("Schema enforcement demo:")
print()

# Try to write with wrong schema (missing column)
wrong_df = df.drop("revenue")
try:
    wrong_df.write.format("delta").mode("append").save(TABLE)
    print("Write succeeded (bad!)")
except Exception as e:
    print(f"❌ Rejected (missing column): {type(e).__name__}")
    print(f"   {str(e)[:100]}...")

# Try to write with incompatible type
wrong_types = df.withColumn("order_id", F.col("order_id").cast("string"))
try:
    wrong_types.write.format("delta").mode("append").save(TABLE)
    print("Write succeeded (bad!)")
except Exception as e:
    print(f"\n❌ Rejected (wrong type): {type(e).__name__}")

# Correct write — same schema
correct = df.limit(100)
correct.write.format("delta").mode("append").save(TABLE)
print(f"\n✅ Correct schema write accepted")
print(f"   Table now has: {spark.read.format('delta').load(TABLE).count():,} rows")

## Step 5 — Isolation: Snapshot Reads

Delta readers always see a consistent snapshot — the table as it was
at the moment the query started. Concurrent writes don't affect
in-progress reads.


In [ ]:
# Show snapshot isolation via version reads
history = DeltaTable.forPath(spark, TABLE).history() \
                    .select("version","operation") \
                    .orderBy("version").collect()

print("All versions in table:")
for row in history:
    count = spark.read.format("delta") \
                 .option("versionAsOf", row["version"]) \
                 .load(TABLE).count()
    print(f"  Version {row['version']}: {count:,} rows  ({row['operation']})")

print()
print("Each version is a consistent snapshot.")
print("A reader at version N always sees exactly the same data,")
print("regardless of what writes happen concurrently.")

## Summary: ACID in Practice

| Property | How Delta implements it | Benefit |
|---|---|---|
| **Atomicity** | Commit only on JSON write to `_delta_log` | No partial writes |
| **Consistency** | Schema enforcement on every write | No corrupt data |
| **Isolation** | Snapshot reads from committed versions | Concurrent safety |
| **Durability** | `_delta_log` as durable write-ahead log | Survives crashes |

### Common ACID failures WITHOUT Delta (plain Parquet)
- Concurrent writers overwrite each other's files
- Failed write leaves partial data visible
- Schema mismatch causes silent corruption
- Readers see in-progress writes mid-operation

**All of these are impossible with Delta Lake.**
